# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display dataset name and description
print(dataset.metadata.name)
print(dataset.metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets by their @id
record_sets = dataset.record_sets
print('Record Sets:')
for rs in record_sets:
    print(f"  @id: {rs.id}, Name: {rs.name}")

# List fields and columns in each record set by their @id
for rs in record_sets:
    print(f"\nFields in Record Set '@id': {rs.id}")
    for field in rs.fields:
        print(f"    Field @id: {field.id}, Name: {field.name}, Data type: {field.data_type}")
        for col in field.columns:
            print(f"      Column @id: {col.id}, Name: {col.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record sets, fields, and columns are referenced by their `@id`.

In [ ]:
# Extract all record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded Record Set @id: {rs_id}, shape: {df.shape}")

# Display column names from the first record set (as example)
first_rs_id = record_set_ids[0]
print(f"\nColumns in Record Set @id: {first_rs_id}")
print(dataframes[first_rs_id].columns.tolist())
dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, categorizing data, and grouping. All references to fields or columns use their `@id`s.

In [ ]:
# Choose a record set and numeric field for EDA
# For example: assume the main survey record set ID is 'https://api.app.sen.science/frontiers/7160411/dataset_main_records'
# and GAD-7 score column @id is 'https://api.app.sen.science/frontiers/7160411/gad7_score',
# and education group column @id is 'https://api.app.sen.science/frontiers/7160411/level_of_education'
# You can replace these with actual @id values from the overview above if different.

# Set these IDs according to the dataset structure:
record_set_id = record_set_ids[0]  # Using the first record set

# Find a numeric field (e.g., GAD-7 score)
# You may want to review the outputs above and choose the @id for a numeric column
numeric_field_id = None
for field in dataset.record_sets[0].fields:
    if field.data_type in ['Float', 'Integer', 'Number'] and len(field.columns) > 0:
        numeric_field_id = field.columns[0].id
        break

print(f"Using Numeric Field @id: {numeric_field_id}")

# Filter DataFrame by numeric column (for demo, threshold=10)
df = dataframes[record_set_id].copy()
if numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records in @{record_set_id} with @{numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized @{numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (e.g., level_of_education)
    group_field_id = None
    for field in dataset.record_sets[0].fields:
        if field.data_type in ['Text'] and len(field.columns) > 0:
            group_field_id = field.columns[0].id
            break

    print(f"Grouping by field @id: {group_field_id}")
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by @{group_field_id} (mean of @{numeric_field_id}):")
        print(grouped_df.head())
else:
    print(f"Numeric field @id {numeric_field_id} not in columns for @{record_set_id}.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All axes and grouping must reference columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If numeric_field_id and group_field_id are found and present
if numeric_field_id and group_field_id and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(8,6))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of field @{numeric_field_id}")
    plt.xlabel(f"@id: {numeric_field_id}")
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(10,6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"Group-wise distribution of @{numeric_field_id} by @{group_field_id}")
    plt.xlabel(f"Group: @{group_field_id}")
    plt.ylabel(f"@id: {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.show()
else:
    print("Cannot plot distributions: required fields not found.")

## 6. Conclusion
Summarize your key findings and observations from the dataset exploration. In this notebook, we loaded and reviewed a mental health survey dataset using the `mlcroissant` library, referencing all entities by their `@id`. Using dynamic extraction, normalization, and visualization, we demonstrated reproducible steps for AI-ready dataset exploration.

Further analysis could expand to additional record sets, more complex groupings, or custom statistical measures. All processing ensures fidelity to Croissant schema `@id` references for reproducible and standards-compliant workflows.